In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
plt.rcParams['figure.dpi'] = 200
plt.rcParams.update({'font.size': 6})
plt.rcParams['figure.figsize'] = [2, 2]

In [ ]:
def process_plate_data(data_dir, wells, rt_wells, value_col, get_well_fn=None, preprocess_fn=None, use_unique_wells=False):
    file_path = os.path.join(data_dir, 'cells.csv')
    df = pd.read_csv(file_path)
    
    if preprocess_fn:
        df = preprocess_fn(df)
    else:
        df = df.dropna()

    if get_well_fn:
        df['Well'] = df['ImageNumber'].apply(lambda x: get_well_fn(x, wells))
    else:
        df['Well'] = df['ImageDir'].apply(lambda x: x.split('/')[6].split('_')[2])

    if use_unique_wells:
        rt_dict = dict(zip(df['Well'].unique(), rt_wells))
    else:
        rt_dict = dict(zip(wells, rt_wells))
        
    df['RT_well'] = df['Well'].map(rt_dict).astype('Int64')
    median_df = df.groupby(['Well', 'RT_well', 'UV_Status'])[value_col].median().reset_index()
    return median_df

def run_experiment(configs, exp_id, value_col):
    dfs = []
    for cfg in configs:
        df = process_plate_data(
            data_dir=cfg['dir'],
            wells=cfg.get('wells', []),
            rt_wells=cfg['rt'],
            value_col=value_col,
            get_well_fn=cfg.get('well_mapper'),
            preprocess_fn=cfg.get('preprocess_fn'),
            use_unique_wells=cfg.get('use_unique_wells', False)
        )
        dfs.append(df)
    concat_df = pd.concat(dfs, ignore_index=True)
    concat_df['Experiment'] = exp_id
    return concat_df

In [ ]:
EXPT1_CONFIG = [
    {
        'dir': '021723_F1-E15.25-nucleolin-total-area-AC/',
        'wells': ['A01', 'A02', 'A03', 'A04', 'A05', 'A06', 'B06', 'B05', 'B04', 'B03', 'B02', 'B01', 'C01', 'C02', 'C03', 'C04', 'C05'],
        'rt': [193, 194, 195, 196, 197, 198, 210, 209, 208, 207, 206, 205, 217, 218, 219, 220, 221],
        'well_mapper': lambda x, w: w[x // 829]
    },
    {
        'dir': '022223_F1-E15.25-nucleolin-total-area-second-plate-D/',
        'wells': ['D06', 'D05', 'D04', 'D03', 'D02', 'D01'],
        'rt': [282, 281, 280, 279, 278, 277],
        'well_mapper': lambda x, w: w[x // 829]
    },
    {
        'dir': '022323_F1-E15.25-nucleolin-total-area-first-plate-DC6/',
        'wells': ['D06', 'D05', 'D04', 'D03', 'D02', 'D01', 'C06'],
        'rt': [234, 233, 232, 231, 230, 229, 222],
        'well_mapper': lambda x, w: w[x // 829]
    },
    {
        'dir': '022623_F1-E15.25-nucleolin-total-area-third-plate-CB/',
        'wells': ['D06', 'D05', 'D04', 'D03', 'D02', 'D01', 'C01', 'C02', 'C03', 'C04', 'C05', 'C06', 'B01'],
        'rt': [240, 239, 238, 237, 236, 235, 223, 224, 225, 226, 227, 228, 211],
        'well_mapper': lambda x, w: w[x // 829] if x < 829*5 else ('D01' if (829*5 <= x < 829*5 + 841) else w[(x-12) // 829])
    }
]

In [ ]:
expt1_df = run_experiment(EXPT1_CONFIG, exp_id=1, value_col='Nucleolar_Area')

In [ ]:
wells_A = ['A01', 'A02', 'A03', 'A04', 'A05', 'A06']
wells_B1_B6 = ['B06', 'B05', 'B04', 'B03', 'B02', 'B01']
wells_C = ['C01', 'C02', 'C03', 'C04', 'C05', 'C06']
wells_D = ['D06', 'D05', 'D04', 'D03', 'D02', 'D01']

EXPT2_CONFIG = [
    {'dir': '040623_F1-E15.25-SiRDNA-plate2-total-area/', 'wells': ['B03', 'B02', 'B01'], 'rt': [339, 338, 337], 'well_mapper': lambda x, w: w[x // 784] if x < 784 else ('B02' if (784 <= x < 784 + 677) else 'B01')},
    {'dir': '040723_F1-E15.25-SiRDNA-plate2-C-total-area/', 'wells': wells_C, 'rt': [349, 350, 351, 352, 353, 354], 'well_mapper': lambda x, w: w[x // 784]},
    {'dir': '040923_F1-E15.25-SiRDNA-plate3-total-area/', 'wells': ['B03', 'B02', 'B01'] + wells_C, 'rt': [375, 374, 373, 295, 296, 297, 298, 299, 300], 'well_mapper': lambda x, w: w[x // 784]},
    {'dir': '041123_F1-E15.25-SiRDNA-plate4-total-area/', 'wells': ['B03', 'B02', 'B01'] + wells_C, 'rt': [321, 320, 319, 331, 332, 333, 334, 335, 336], 'well_mapper': lambda x, w: w[x // 784]},
    {'dir': '041323_F1-E15.25-SiRDNA-plate5-total-area/', 'wells': ['B03', 'B02', 'B01'] + wells_C, 'rt': [357, 356, 355, 367, 368, 369, 370, 371, 372], 'well_mapper': lambda x, w: w[x // 784]},
    {'dir': '041623_F1-E15.25-SiRDNA-plate6-total-area/', 'wells': wells_D + wells_C, 'rt': [318, 317, 316, 315, 314, 313, 301, 302, 303, 304, 305, 306], 'well_mapper': lambda x, w: w[x // 784]},
    {'dir': '040523_F1-E15.25-nucleolin-plate2-total-area/', 'wells': wells_A + ['B06', 'B05', 'B04'], 'rt': [325, 326, 327, 328, 329, 330, 342, 341, 340], 'well_mapper': lambda x, w: w[x // 784]},
    {'dir': '040823_F1-E15.25-nucleolin-plate3-total-area/', 'wells': wells_A + ['B06', 'B05', 'B04'], 'rt': [361, 362, 363, 364, 365, 366, 378, 377, 376], 'well_mapper': lambda x, w: w[x // 784]},
    {'dir': '041023_F1-E15.25-nucleolin-plate4-total-area-A/', 'wells': wells_A, 'rt': [307, 308, 309, 310, 311, 312], 'well_mapper': lambda x, w: w[x // 784]},
    {'dir': '041223_F1-E15.25-nucleolin-plate5-total-area/', 'wells': wells_A, 'rt': [343, 344, 345, 346, 347, 348], 'well_mapper': lambda x, w: w[x // 784]},
    {'dir': '041423_F1-E15.25-nucleolin-plate6-total-area/', 'wells': wells_A + wells_B1_B6, 'rt': [379, 380, 381, 382, 383, 384, 294, 293, 292, 291, 290, 289], 'well_mapper': lambda x, w: w[x // 784]},
    {'dir': '041623_F1-E15.25-nucleolin-plate4-B-total-area/', 'wells': ['B06', 'B05', 'B04'], 'rt': [324, 323, 322], 'well_mapper': lambda x, w: w[x // 784]},
    {'dir': '041623_F1-E15.25-nucleolin-plate5-B-total-area/', 'wells': ['B06', 'B05'], 'rt': [360, 359], 'well_mapper': lambda x, w: w[x // 784]},
    {'dir': '041823_F1-E15.25-nucleolin-plate5-B04-total-area/', 'wells': ['B04'], 'rt': [358], 'well_mapper': lambda x, w: w[x // 784]},
]

In [ ]:
expt2_df = run_experiment(EXPT2_CONFIG, exp_id=2, value_col='Nucleolar_Area')

In [ ]:
def exp3_preprocess(df):
    df['UV_Status'] = df['Nuclear_Area']
    df['Nuclear_Area'] = df['Nucleolar_Area'] / df['Nucleolar_Ratio']
    return df

In [ ]:
EXPT3_AREA_CFG = [
    {'dir': '041823_F1-E15.25-SC35-plate1-total-area/', 'rt': [289, 290, 291, 292, 293, 294, 306], 'preprocess_fn': ex3_preprocess, 'use_unique_wells': True},
    {'dir': '042023_F1-E15.25-SC35-plate1C-total-area/', 'rt': [315, 316, 317, 318], 'use_unique_wells': True},
    {'dir': '042123_F1-E15.25-SC35-plate2-total-area/', 'rt': [325, 326, 327, 328, 329, 330, 342, 341, 340], 'use_unique_wells': True},
    {'dir': '042523_F1-E15.25-SC35-plate4-total-area/', 'rt': [331, 332, 333, 334, 335, 336], 'use_unique_wells': True},
    {'dir': '042723_F1-E15.25-SC35-plate3-total-area-C/', 'rt': [323, 324], 'use_unique_wells': True},
    {'dir': '042723_F1-E15.25-SC35-plate4-total-area-C/', 'rt': [355, 356, 357, 358], 'use_unique_wells': True},
]

EXPT3_FRAC_CFG = [
    {'dir': '041923_F1-E15.25-SC35-plate1-ratio/', 'rt': [305, 304, 303, 302, 301, 313, 314], 'preprocess_fn': ex3_preprocess, 'use_unique_wells': True},
    {'dir': '042223_F1-E15.25-SC35-plate2-ratio/', 'rt': [339, 338, 337, 349, 350, 351, 352, 353, 354], 'use_unique_wells': True},
    {'dir': '042423_F1-E15.25-SC35-plate3-ratio/', 'rt': [310, 309, 308, 307, 319, 320, 321, 322], 'use_unique_wells': True},
    {'dir': '042623_F1-E15.25-SC35-plate4-ratio/', 'rt': [348, 347, 346, 345, 344, 343], 'use_unique_wells': True},
    {'dir': '042823_F1-E15.25-SC35-plate4-ratio-C/', 'rt': [359], 'use_unique_wells': True},
    {'dir': '042823_F1-E15.25-SC35-plate4-ratio-C06/', 'rt': [360], 'use_unique_wells': True},
]

In [ ]:
expt3_area_df = run_experiment(EXPT3_AREA_CFG, exp_id=3, value_col='Nucleolar_Area')
expt3_frac_df = run_experiment(EXPT3_FRAC_CFG, exp_id=3, value_col='Nucleolar_Ratio')

In [ ]:
## Add plate3 manually
plate3_raw = pd.read_csv('042423_F1-E15.25-SC35-plate3-ratio/cells.csv').dropna()
plate3_raw['Well'] = plate3_raw['ImageDir'].apply(lambda x: x.split('/')[6].split('_')[2])
plate3_raw['Col'] = plate3_raw['Well'].apply(lambda x: x[1:3])

conds = [
    (plate3_raw['Nucleolar_Area'] < 848),
    (plate3_raw['Nucleolar_Area'] >= 848) & (plate3_raw['Nucleolar_Area'] < 1049),
    (plate3_raw['Nucleolar_Area'] >= 1049) & (plate3_raw['Nucleolar_Area'] < 1272),
    (plate3_raw['Nucleolar_Area'] >= 1272)
]

plate3_raw['UV_Status'] = np.select(conds, ['Q1', 'Q2', 'Q3', 'Q4'])
plate3_area_sub = plate3_raw.groupby(['Col', 'UV_Status'])['Nucleolar_Area'].median().reset_index()

col56_df = pd.read_csv('042723_F1-E15.25-SC35-plate3-total-area-C/cells.csv').dropna()
col56_df['Well'] = col56_df['ImageDir'].apply(lambda x: x.split('/')[6].split('_')[2])
rt_dict_col56 = dict(zip(col56_df['Well'].unique(), [323, 324]))
col56_df['RT_well'] = col56_df['Well'].map(rt_dict_col56).astype('Int64')
col56 = col56_df.groupby(['Well', 'RT_well', 'UV_Status'])['Nucleolar_Area'].median().reset_index()
col56['Col'] = col56['Well'].apply(lambda x: x[1:3])
col56 = col56.drop(['Well', 'RT_well'], axis=1)

plate3_joined = pd.concat([plate3_area_sub, col56])
plate3_missing = pd.DataFrame({'Well': ['A01', 'A02', 'A03', 'A04', 'A05', 'A06', 'B06', 'B05'], 'RT_well': [295, 296, 297, 298, 299, 300, 312, 311], 'key': 1})
bin_df = pd.DataFrame(['Q1', 'Q2', 'Q3', 'Q4'], columns=['UV_Status']).assign(key=1)

plate3_missing = pd.merge(plate3_missing, bin_df, on='key').drop('key', axis=1)
plate3_missing['Col'] = plate3_missing['Well'].apply(lambda x: x[1:3])
plate3_missing = pd.merge(plate3_missing, plate3_joined, on=['Col', 'UV_Status']).drop('Col', axis=1).sort_values(['Well', 'UV_Status']).reset_index(drop=True)
plate3_missing['Experiment'] = 3

In [ ]:
expt3_area_df = pd.concat([expt3_area_df, plate3_missing], ignore_index=True)
expt3_frac_df = expt3_frac_df.rename(columns={'Nucleolar_Ratio': 'Nucleolar_Area'})
expt3_df = pd.concat([expt3_area_df, expt3_frac_df], ignore_index=True)

In [ ]:
EXPT4_AREA_CFG = [
    {'dir': '070723_F1-E15.25-nuclear-size-plate1/', 'rt': [289, 290, 291, 292, 293, 294, 306, 305, 304], 'use_unique_wells': True},
    {'dir': '070923_F1-E15.25-nuclear-size-plate2/', 'rt': [325, 326, 327, 328, 329, 330, 342, 341, 340], 'use_unique_wells': True},
    {'dir': '071323_F1-E15.25-nuclear-size-plate3/', 'rt': [295, 296, 297, 298, 299, 300, 312, 311, 310], 'use_unique_wells': True},
    {'dir': '071523_F1-E15.25-nuclear-size-plate4/', 'rt': [331, 332, 333, 334, 335, 336, 348, 347, 346], 'use_unique_wells': True},
    {'dir': '071723_F1-E15.25-nuclear-size-plate5/', 'rt': [361, 362, 363, 364, 365, 366, 372, 371, 370], 'use_unique_wells': True},
]

EXPT4_FRAC_CFG = [
    {'dir': '070823_F1-E15.25-nucleolar-fraction-plate1/', 'rt': [303, 302, 301, 313, 314, 315, 316, 317, 318], 'use_unique_wells': True},
    {'dir': '071023_F1-E15.25-nucleolar-fraction-plate2/', 'rt': [339, 338, 337, 349, 350, 351, 352, 353, 354], 'use_unique_wells': True},
    {'dir': '071223_F1-E15.25-nucleolar-fraction-plate3/', 'rt': [309, 308, 307, 319, 320, 321, 322, 323, 324], 'use_unique_wells': True},
    {'dir': '071423_F1-E15.25-nucleolar-fraction-plate4/', 'rt': [345, 344, 343, 355, 356, 357, 358, 359, 360], 'use_unique_wells': True},
    {'dir': '071623_F1-E15.25-nucleolar-fraction-plate5/', 'rt': [369, 368, 367, 373, 374, 375, 376, 377, 378], 'use_unique_wells': True},
]

In [ ]:
expt4_area_df = run_experiment(EXPT4_AREA_CFG, exp_id=4, value_col='Nuclear_Area')
expt4_area_df = expt4_area_df.rename(columns={'Nuclear_Area': 'Nucleolar_Area'})

expt4_frac_df = run_experiment(EXPT4_FRAC_CFG, exp_id=4, value_col='Nucleolar_Fraction')
expt4_frac_df = expt4_frac_df.rename(columns={'Nucleolar_Fraction': 'Nucleolar_Area'})

expt4_df = pd.concat([expt4_area_df, expt4_frac_df], ignore_index=True)

In [ ]:
expts_concat = pd.concat([expt1_df, expt2_df, expt3_df, expt4_df])
expts_concat = expts_concat.rename(columns={"Nucleolar_Area": "Bin_Median", "UV_Status": "Bin"})

In [ ]:
expts_concat

In [ ]:
expts_concat.to_csv('expts_phenotype_median_df_refactored.txt', index=False)

In [ ]:
expts_concat